In [ ]:
# Fake News Detection - Model Comparison & Benchmarks
# This notebook compares multiple Hugging Face models for fake news detection

import os
import json
import time
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import classification_report, confusion_matrix, roc_curve, auc
from transformers import pipeline, AutoTokenizer, AutoModelForSequenceClassification
import torch

# Set up environment
from dotenv import load_dotenv
load_dotenv()

print("Libraries loaded successfully!")

In [ ]:
# Define models to compare
MODELS_TO_COMPARE = [
    {
        "name": "BERT Tiny (Current)",
        "model_id": "mrm8488/bert-tiny-finetuned-fake-news-detection",
        "size_mb": 16,
        "type": "text-classification"
    },
    {
        "name": "RoBERTa OpenAI Detector",
        "model_id": "roberta-base-openai-detector",
        "size_mb": 500,
        "type": "text-classification"
    },
    {
        "name": "DeBERTa v3 Base",
        "model_id": "microsoft/deberta-v3-base",
        "size_mb": 710,
        "type": "text-classification"
    },
    {
        "name": "DistilBERT Base",
        "model_id": "distilbert-base-uncased",
        "size_mb": 268,
        "type": "text-classification"
    }
]

# Sample test dataset (in production, load from real datasets like Liar, FakeNewsNet)
TEST_SAMPLES = [
    {
        "text": "Breaking: Scientists discover miracle cure for all diseases using kitchen ingredients!",
        "label": "fake",
        "category": "health_misinformation"
    },
    {
        "text": "The government has secretly been controlling the weather for years, leaked documents show.",
        "label": "fake",
        "category": "conspiracy"
    },
    {
        "text": "Local elections will be held next month with three candidates running for mayor.",
        "label": "real",
        "category": "politics"
    },
    {
        "text": "You won't believe what this celebrity did last night - shocking details inside!",
        "label": "fake",
        "category": "clickbait"
    },
    {
        "text": "The stock market closed higher today as tech stocks rallied on positive earnings reports.",
        "label": "real",
        "category": "finance"
    },
    {
        "text": "NASA confirms Earth is actually flat and space travel is a hoax created by Hollywood.",
        "label": "fake",
        "category": "conspiracy"
    },
    {
        "text": "Researchers find that moderate coffee consumption may have health benefits.",
        "label": "real",
        "category": "health"
    },
    {
        "text": "Click here to claim your free $1000 gift card before time runs out!",
        "label": "fake",
        "category": "scam"
    }
]

print(f"Test dataset loaded with {len(TEST_SAMPLES)} samples")
print(f"Models to compare: {len(MODELS_TO_COMPARE)}")

In [ ]:
# Load and benchmark models
results = []
model_pipelines = {}

for model_info in MODELS_TO_COMPARE:
    print(f"\nLoading {model_info['name']}...")
    start_time = time.time()
    
    try:
        # Load the model pipeline
        hf_token = os.getenv("HF_TOKEN")
        pipeline_kwargs = {"model": model_info["model_id"]}
        if hf_token:
            pipeline_kwargs["token"] = hf_token
            
        pipe = pipeline(
            model_info["type"],
            **pipeline_kwargs,
            truncation=True,
            max_length=512
        )
        
        load_time = time.time() - start_time
        model_pipelines[model_info["name"]] = pipe
        print(f"✓ Loaded in {load_time:.2f}s")
        
        # Benchmark inference time
        inference_times = []
        predictions = []
        
        for sample in TEST_SAMPLES:
            inf_start = time.time()
            result = pipe(sample["text"][:500])
            inf_time = time.time() - inf_start
            inference_times.append(inf_time)
            
            # Extract prediction
            if isinstance(result, list) and len(result) > 0:
                top_result = result[0]
                label = str(top_result.get("label", ""))
                score = float(top_result.get("score", 0.5))
                
                # Normalize label to fake/real
                is_fake = "fake" in label.lower() or "unreliable" in label.lower() or label == "1"
                predictions.append({
                    "predicted": is_fake,
                    "confidence": score,
                    "raw_label": label
                })
            else:
                predictions.append({"predicted": False, "confidence": 0.5, "raw_label": "unknown"})
        
        # Calculate metrics
        avg_inference_time = np.mean(inference_times)
        
        # Calculate accuracy
        correct = sum(1 for i, pred in enumerate(predictions) 
                     if pred["predicted"] == (TEST_SAMPLES[i]["label"] == "fake"))
        accuracy = correct / len(TEST_SAMPLES)
        
        results.append({
            "model": model_info["name"],
            "model_id": model_info["model_id"],
            "size_mb": model_info["size_mb"],
            "load_time_s": load_time,
            "avg_inference_ms": avg_inference_time * 1000,
            "accuracy": accuracy,
            "predictions": predictions
        })
        
        print(f"  Accuracy: {accuracy:.2%}")
        print(f"  Avg inference: {avg_inference_time*1000:.1f}ms")
        
    except Exception as e:
        print(f"✗ Failed to load: {e}")
        results.append({
            "model": model_info["name"],
            "model_id": model_info["model_id"],
            "size_mb": model_info["size_mb"],
            "load_time_s": None,
            "avg_inference_ms": None,
            "accuracy": None,
            "error": str(e)
        })

print(f"\nBenchmark complete! Results for {len(results)} models.")

In [ ]:
# Create comparison dataframe
df_results = pd.DataFrame([r for r in results if "error" not in r])
if not df_results.empty:
    # Display results table
    display_cols = ["model", "size_mb", "load_time_s", "avg_inference_ms", "accuracy"]
    print("\n=== MODEL COMPARISON RESULTS ===")
    print(df_results[display_cols].to_string(index=False))
    
    # Calculate efficiency score (accuracy / (size_mb / 100))
    df_results["efficiency_score"] = df_results["accuracy"] / (df_results["size_mb"] / 100)
    print("\n=== EFFICIENCY RANKING ===")
    print(df_results[["model", "accuracy", "size_mb", "efficiency_score"]].sort_values("efficiency_score", ascending=False).to_string(index=False))
else:
    print("No successful model results to display.")

In [ ]:
# Visualize results
if not df_results.empty:
    fig, axes = plt.subplots(2, 2, figsize=(14, 10))
    fig.suptitle('Fake News Detection Model Comparison', fontsize=16)
    
    # Accuracy comparison
    axes[0, 0].bar(df_results["model"], df_results["accuracy"], color='skyblue')
    axes[0, 0].set_title('Accuracy Comparison')
    axes[0, 0].set_ylabel('Accuracy')
    axes[0, 0].tick_params(axis='x', rotation=45)
    
    # Inference time comparison
    axes[0, 1].bar(df_results["model"], df_results["avg_inference_ms"], color='lightcoral')
    axes[0, 1].set_title('Average Inference Time (ms)')
    axes[0, 1].set_ylabel('Time (ms)')
    axes[0, 1].tick_params(axis='x', rotation=45)
    
    # Model size vs accuracy scatter
    axes[1, 0].scatter(df_results["size_mb"], df_results["accuracy"], s=100, alpha=0.7)
    for idx, row in df_results.iterrows():
        axes[1, 0].annotate(row["model"], (row["size_mb"], row["accuracy"]), 
                          xytext=(5, 5), textcoords='offset points', fontsize=8)
    axes[1, 0].set_xlabel('Model Size (MB)')
    axes[1, 0].set_ylabel('Accuracy')
    axes[1, 0].set_title('Model Size vs Accuracy Trade-off')
    axes[1, 0].grid(True, alpha=0.3)
    
    # Load time comparison
    axes[1, 1].bar(df_results["model"], df_results["load_time_s"], color='lightgreen')
    axes[1, 1].set_title('Model Load Time (s)')
    axes[1, 1].set_ylabel('Time (s)')
    axes[1, 1].tick_params(axis='x', rotation=45)
    
    plt.tight_layout()
    plt.show()
else:
    print("No data to visualize")